In [ ]:
!pip install langchain-huggingface

In [ ]:
!pip install langchain

In [ ]:
API_KEY ="Huggingface API Key"

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_core.prompts import PromptTemplate


llm = HuggingFaceEndpoint(
    repo_id="prism-ml/Ternary-Bonsai-27B-gguf",
    task="text-generation",
    huggingfacehub_api_token=API_KEY
)

model = ChatHuggingFace(llm=llm)

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

prompt1 = template1.invoke({'topic':'black hole'})

result = model.invoke(prompt1)

prompt2 = template2.invoke({'text':result.content})

result1 = model.invoke(prompt2)

print(result1.content)

In [ ]:
!pip install -U langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [ ]:
!pip install langchain

In [ ]:

google_api_key = "ADD YOUR GOOGLE API KEY HERE"


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)


Here is a 5-line summary of the text:

1. **Definition & History:** Black holes are regions of spacetime where gravity is so intense that nothing can escape, having evolved from theoretical relativity concepts into proven cosmological realities.
2. **Structure & Types:** Categorized by mass from stellar to supermassive scales, black holes feature distinct regions including a central singularity, an event horizon, and surrounding accretion disks.
3. **Physical Phenomena:** Objects approaching a black hole experience extreme relativistic effects, such as gravitational time dilation, tidal spaghettification, and potential evaporation via Hawking radiation.
4. **Observational Methods:** Because black holes emit no light, scientists observe them through gravitational wave collisions, stellar orbital dynamics, and direct shadow imaging via the Event Horizon Telescope.
5. **Cosmic Significance:** Beyond shaping galaxy formation, black holes highlight unresolved frontiers in physics, particula

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)


parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)


{'topic': 'Black Holes', 'facts': ['Nothing, not even light, can escape a black hole once it crosses its boundary known as the event horizon.', 'Due to extreme gravitational time dilation, time moves slower near a black hole relative to an observer further away.', "An object falling into a black hole experiences 'spaghettification,' where intense tidal forces stretch it vertically and compress it horizontally.", 'Supermassive black holes, which can be millions or billions of times the mass of our Sun, reside at the centers of most galaxies, including the Milky Way.', 'Black holes are not completely black; they theoretically emit small amounts of thermal radiation, known as Hawking radiation, which can cause them to slowly evaporate over time.']}


In [ ]:
# structured outut parser

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StructuredOutputParser, ResponseSchema

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

schema = [
    ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
    ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
    ResponseSchema(name='fact_3', description='Fact 3 about the topic'),
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template='Give 3 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)

name='Kavinda Fernando' age=28 city='Kandy'
